# NB-17-JAO · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-17-JAO**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-17-JAO.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This is an observed auction-record audit alongside the controlled FTR capstone. It checks bids, awards, and delivery products for one Nemo Link auction; it does not assume those transmission-capacity products are US FTRs or reconstruct the auction engine.

- [Volume V, Chapter 13 — Representing rights, auctions, exposures, and settlements](https://sherandoresearch.github.io/power-markets-series/vol5_representing_power_markets.html#c13)
- [Volume VI, Chapter 12 — Bid books and uncertain awards](https://sherandoresearch.github.io/power-markets-series/vol6_inference_and_decisions.html#c12)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-17-JAO', require_key=True)
print(setup_status)


# NB-17-JAO — Observed bids, awards, and delivery products
Companion to NB-17's controlled bid-book capstone. Here the question is narrower:
**can published bids and awards be reconciled to one observed auction?**
We use actual Nemo Link BE→GB intraday records for 1 September 2026, retained
from EWB's JAO snapshot `20261015T140704Z`. No synthetic data or hypothetical
bids are introduced. This is an audit of published outcomes, not an auction engine.

Work through capture selection, calendar validation, product-level reconciliation,
and the difference between an observed bid curve and a counterfactual allocation.
Execution outputs remain in the private run folder; rerunning requires the
authenticated private methods service and sealed observations.

In [ ]:
NOTEBOOK_ID = 'NB-17-JAO'

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return response['result']
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

print('Local raw capture inspection:',inputs['raw']['bid_pages'][:1])
remote=execute('auction','auction_record_reconciliation',inputs['requests']['auction'])['auction_audit']
assert remote['rights_type']=='unspecified_in_retained_capture'
j=dict(audit=dict(inputs['audit']),auction=inputs['auction'],bids=inputs['selected_bids'],curtailments=inputs['raw']['curtailments'],products=[])
for p in remote['products']:
    j['products'].append(dict(product=p['product_id'],hour=inputs['product_labels'][p['product_id']],
        start=datetime.fromisoformat(p['start']),end=datetime.fromisoformat(p['end']),hours=p['duration_hours'],
        offered=p['offered'],requested=p['requested'],awarded=p['awarded'],price=p['price'],charge=p['capacity_charge'],
        bids=[r for r in inputs['selected_bids'] if r['product_identification']==p['product_id']]))
j['audit']['awarded_MWh']=sum(p['awarded']*p['hours'] for p in j['products'])
j['audit']['capacity_charge_EUR']=sum(p['charge'] for p in j['products'])
print('Auction:',inputs['auction_id'])
print(json.dumps(j['audit'],indent=2))

## 1. One compatible capture set, not all rows stacked together
The auction endpoint returns ten captures and the bid endpoint three. We select
one auction payload by latest source update, then latest collector receipt, then
hash, and require result rows from that **same auction payload**. The bid endpoint
has a separate payload: apply the same deterministic selection within bids, keep
all its rows, and require exact product coverage plus reconciled requested and
awarded quantities and awarded prices. No cross-payload deduplication by bid index.

The auction/result update is 10:00:20.191 UTC on 2 September; selected bid rows
have updates 10:00:22.711 or .712 UTC. These are not equal revision identifiers.
We establish **economic compatibility for this capture set**, not a transactional
snapshot guarantee. Hashes and timestamps remain distinct in the audit above.
Receipt time is collector knowledge, not historical publication time. This is a
retrospective latest-view audit, not evidence of information available at bid time.
`bid_index` is a position inside one payload, never a participant identifier.

## 2. Delivery calendar: six hourly products, not six copies of the start
The auction covers 04:00–10:00 UTC. Product labels run 06:00–12:00 in local
CET/CEST time. The [Nemo Link access rules](https://www.jao.eu/sites/default/files/news_media/Nemo%20Link%20Access%20Rules%20v4_January%202023.pdf)
define the local contract-day convention; the [operator's intraday description](https://www.nemolink.co.uk/category/intraday/)
describes auctions spanning up to six hours. For this non-transition date, we
interpret labels with `Europe/Brussels`, then verify the UTC conversion against
the observed start **and stop**, product order, and every bid's 60 delivered minutes.
This is a sample-specific calendar check, not a universal current-rule assertion.

Six contiguous intervals exactly tile the auction window. `market_period_start`
alone is insufficient. `period_to_be_secured_*` is not the delivery window.
The decoder deliberately rejects other dates; DST and subhourly extensions need
their own validated calendars rather than silently reusing this one.

In [ ]:
print('Product | UTC interval | offered MW | requested MW | awarded MW | EUR/MWh | capacity charge EUR')
for p in j['products']:
    print(p['product'],p['start'].strftime('%H:%M')+'–'+p['end'].strftime('%H:%M'),
          p['offered'],p['requested'],p['awarded'],p['price'],round(p['charge'],2),sep=' | ')
print('Awarded capacity-hours:',j['audit']['awarded_MWh'],'MW h')
print('Awarded-volume × published-price × duration:',round(j['audit']['capacity_charge_EUR'],2),'EUR')

## 3. Reconciliation before interpretation
In each product, bid quantities sum to published requested capacity and awarded
quantities sum to published allocated capacity. Positive awards have the product's
published auction price, are nonnegative, and do not exceed the corresponding
bid quantity. Awarded capacity does not exceed offered capacity. The second
product allocates 1,581 of 1,582 MW: we retain the one-MW difference and do not
force an artificial full allocation or infer its cause.

Capacity-hours below measure awarded transmission rights, not metered energy
delivered. Multiplying awards by their prices and durations gives EUR 1,566.01
in capacity charges before other charges or adjustments. It is not a verified
invoice, realized energy-trading profit, or settlement payoff. No spot spread,
nominations, losses, fees, or participant portfolio is available in this audit.

In [ ]:
plt=setup_plot();labels=[p['hour'] for p in j['products']]
fig,axes=plt.subplots(1,2,figsize=(12,4))
x=np.arange(6)
for offset,key,color in [(-.25,'requested','#B48EAD'),(0,'offered','#88C0D0'),(.25,'awarded','#A3BE8C')]:
    axes[0].bar(x+offset,[p[key] for p in j['products']],width=.25,label=key,color=color)
axes[0].set(ylabel='Capacity · MW',title='Observed product totals');axes[0].legend()
axes[1].bar(x,[p['charge'] for p in j['products']],color='#EBCB8B')
axes[1].set(ylabel='Capacity charge · EUR',title='Awarded MW × EUR/MWh × 1 hour')
for ax in axes:ax.set_xticks(x,labels,rotation=35,ha='right');ax.set_xlabel('Product hours · CEST')
fig.tight_layout();plt.show()

## 4. Observed curves do not identify counterfactual awards
Sort positive-quantity rows by submitted price to display each observed demand
staircase. Its vertical marker is the **published awarded volume**, not a computed
clearing point; the horizontal marker is the published price. Zero-quantity rows
stay in the retained capture and reconciliation but add no curve width.
The plot describes submitted quantities, not a new bid or market simulation.

A changed bid might alter price, tie handling, credit validation, or allocation.
This export does not identify those mechanisms. We do not run PJM simultaneous
feasibility, import PJM PTDFs, infer individual traders, or compute counterfactual
awards. Identical bids can be distinct submitted rows; the payload-local index
cannot be used to stitch participants across products or captures.

In [ ]:
fig,axes=plt.subplots(2,3,figsize=(13,7))
for ax,p in zip(axes.flat,j['products']):
    bids=sorted((b for b in p['bids'] if b['quantity']>0),key=lambda b:(-b['price'],b['bid_index']))
    quantity=np.array([b['quantity'] for b in bids]);price=np.array([b['price'] for b in bids])
    edges=np.r_[0,np.cumsum(quantity)]
    ax.stairs(price,edges,color='#88C0D0',label='Observed bids')
    ax.axvline(p['awarded'],color='#A3BE8C',linestyle='--',label='Awarded MW')
    ax.axhline(p['price'],color='#EBCB8B',linestyle=':',label='Auction price')
    ax.set(title=p['hour']+' CEST',xlabel='Cumulative requested MW',ylabel='Bid price · EUR/MWh');ax.grid(alpha=.25)
axes[0,0].legend(fontsize=8);fig.tight_layout();plt.show()

## 5. Curtailment scope and next evidence
A fully paginated corridor query for curtailments starting on 1 September through
00:00 UTC on 2 September returns zero rows. That is **not** proof of no applicable
curtailment: an earlier-starting event could overlap, publication can be revised,
and horizon applicability must be checked. We make no curtailment adjustment and
do not turn the empty result into a zero-risk assumption. Curtailment compensation
and future revisions are outside this snapshot's accounting scope.

Exercise: why would summing all three bid payloads be wrong? They are repeated
captures of the auction, not three auctions. Why is the six-hour window not the
duration of every bid? Each row belongs to a specific one-hour product. The six
intervals and their separate reconciliations above provide the answer scaffold.
A historical strategy study would additionally require admissible earlier
vintages, participant scope, complete event overlap coverage, and an independently
validated allocation model. NB-17's controlled capstone supplies none of these
missing empirical contracts merely by sharing the word “auction.”

In [ ]:
bad=copy.deepcopy(inputs['requests']['auction'])
chosen=bad['inputs']['revision_policy']['bid_payload']
index=next(i for i,b in enumerate(bad['inputs']['auction_records']['bids']) if b['payload']==chosen and b['requested']>0)
bad['inputs']['auction_records']['bids'].pop(index);reject('auction_record_reconciliation',bad)
bad=copy.deepcopy(inputs['requests']['auction'])
row=next(b for b in bad['inputs']['auction_records']['bids'] if b['payload']==chosen)
bad['inputs']['auction_records']['bids'].append(copy.deepcopy(row));reject('auction_record_reconciliation',bad)
summary_products=[{k:v.isoformat() if isinstance(v,datetime) else v for k,v in p.items() if k!='bids'} for p in j['products']]
(session.output/'consumer-summary.json').write_text(json.dumps(dict(audit=j['audit'],products=summary_products),indent=2,allow_nan=False)+'\n')
summary=session.finish({'observed_only':True,'one_capture_per_endpoint':True,'economic_compatibility_reconciled':True,
    'six_product_calendar_validated':True,'award_quantities_and_prices_reconciled':True,'no_participant_inference':True,
    'no_counterfactual_awards':True,'new_synthetic_data':False},
    {'auction':inputs['auction_id'],'calendar_scope':'2026-09-01 only; Europe/Brussels labels verified against UTC bounds'})